# **Licenciatura em Ciências da Computação**

### Aprendizagem Computacional 25/26

# IMDB Sentiment Analysis: Feature Engineering Challenge

## 1. Setup & Data Loading

First, we'll grab the dataset. We'll use a subset (10,000 rows) to keep the processing fast during class.

In [6]:
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score
)

from scipy.stats import ttest_rel


# Load dataset (directly from a common source)
url = "https://raw.githubusercontent.com/Ankit152/IMDB-Sentiment-Analysis/master/IMDB-Dataset.csv"
df = pd.read_csv(url).sample(10000, random_state=42) # Subset for speed

# Encode target
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

print(f"Dataset loaded with {df.shape[0]} reviews.")
df.head()



Dataset loaded with 10000 reviews.


,review,sentiment,label
33553,I really liked this Summerslam due to the look...,positive,1
9427,Not many television shows appeal to quite as m...,positive,1
199,The film quickly gets to a major chase scene w...,negative,0
12447,Jane Austen would definitely approve of this o...,positive,1
39489,Expectations were somewhat high for me when I ...,negative,0


In [7]:
df['label'].value_counts()

label
1    5039
0    4961
Name: count, dtype: int64

## 2. The "Baseline" Model
To know if our features are actually good, we need a baseline. This uses a simple Bag of Words (BoW) approach.

In [8]:
# # Simple vectorization
# vectorizer = CountVectorizer(max_features=1000)
# X_baseline = vectorizer.fit_transform(df['review'])
# y = df['label']

# # Split
# X_train, X_test, y_train, y_test = train_test_split(X_baseline, y, test_size=0.2, random_state=42)

# # Train a simple Logistic Regression
# baseline_model = LogisticRegression(max_iter=1000)
# baseline_model.fit(X_train, y_train)

# # Evaluate
# preds = baseline_model.predict(X_test)
# print(f"Baseline F1-Score: {f1_score(y_test, preds):.4f}")

X = df["review"].astype(str)
y = df["label"].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train size:", len(X_train))
print("Test size :", len(X_test))

Train size: 8000
Test size : 2000


## 3. The Challenge: Feature Engineering Sandbox
Your goal is to create a function that extracts new numerical features from the raw text. Think about:

Metadata: Length, punctuation, capitalization.

Lexicons: Positive/Negative word counts.

Context: Handling "not" or "never."

In [9]:
import string

def extract_custom_features(text):
    """
    Students: Edit this function to create your features!
    Return a list or numpy array of numbers.
    """
    features = []

    # Example Feature 1: Word Count
    words = text.split()
    features.append(len(words))

    # Example Feature 2: Count of Exclamation Marks
    features.append(text.count('!'))

    # Example Feature 3: 'No/Not' density (Simple Negation)
    negations = len(re.findall(r'\b(not|no|never|neither|nor|worse|bad)\b', text.lower()))
    features.append(negations)
    
    good = len(re.findall(r'\b(awsome|good|great|execellent|amazing|won|better|best)\b', text.lower()))
    features.append(good)
    # --- ADD YOUR OWN IDEAS BELOW ---

    return features

def simple_clean(text):
    # Convert to lowercase
    text = text.lower()
    # Standardize whitespace (replace multiple whitespaces with a single space)
    text = re.sub(r'\s+', ' ', text).strip()
    # remove punctuation
    # text = text.translate(str.maketrans('', '', string.punctuation))
    # remove digits
    text = re.sub(r'\d+', '', text)
    return text
# # Apply your features to the dataframe
# custom_features_list = df['review'].apply(extract_custom_features).tolist()
# X_custom = np.array(custom_features_list)

# print(f"New feature matrix shape: {X_custom.shape}")

Additional Tips:

- Check Stop words
- TF-IDF

In [ ]:

X_train_cleaned = X_train.apply(simple_clean)
print("First 5 entries of X_train_cleaned:")
print(X_train_cleaned.head())

X_train_manual_features = pd.DataFrame(X_train.apply(extract_custom_features).tolist())
print("First 5 entries of X_train_manual_features:")
print(X_train_manual_features.head())

vectorizer = CountVectorizer()
X_train_bow = vectorizer.fit_transform(X_train_cleaned)

# Convert the sparse matrix to a DataFrame for easier inspection
X_train_bow_df = pd.DataFrame(
    X_train_bow.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=X_train_cleaned.index
)

print("First 5 entries of X_train_bow_df:")
print(X_train_bow_df.head())

X_train_manual_features = X_train.apply(extract_custom_features).apply(pd.Series)

X_train_combined = pd.concat([X_train_manual_features, X_train_bow_df], axis=1)

X_train_combined.columns = X_train_combined.columns.astype(str)

print("Shape of X_train_combined:", X_train_combined.shape)
print("\nFirst 5 entries of X_train_combined:")
print(X_train_combined.head())

k_features = int(0.75 * X_train_combined.shape[1])

selector = SelectKBest(chi2, k=k_features)
selector.fit(X_train_combined, y_train)

X_train_selected = selector.transform(X_train_combined)

selected_features = X_train_combined.columns[selector.get_support(indices=True)]
X_train_selected = pd.DataFrame(
    X_train_selected,
    columns=selected_features,
    index=X_train_combined.index
)

print(f"Original number of features: {X_train_combined.shape[1]}")
print(f"Number of features selected (k): {k_features}")
print(f"Shape of X_train_selected: {X_train_selected.shape}")
print("\nFirst 5 entries of X_train_selected:")
print(X_train_selected.head())


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def f1_spam(y_true, y_pred):
    return f1_score(y_true, y_pred, pos_label='spam', average='binary')

mnb_f1_scores = []
mnb = MultinomialNB()

# Reset index of X_train_selected and y_train to ensure correct slicing
# This is important if they lost their original index alignment during previous operations
X_train_selected_reset = X_train_selected.reset_index(drop=True)
y_train_reset = y_train.reset_index(drop=True)

for train_index, val_index in skf.split(X_train_selected_reset, y_train_reset):
    X_train_fold, X_val_fold = X_train_selected_reset.iloc[train_index], X_train_selected_reset.iloc[val_index]
    y_train_fold, y_val_fold = y_train_reset.iloc[train_index], y_train_reset.iloc[val_index]
    
    mnb.fit(X_train_fold, y_train_fold)
    y_pred_mnb = mnb.predict(X_val_fold)

    score = f1_spam(y_val_fold, y_pred_mnb)
    mnb_f1_scores.append(score)

print(f"MultinomialNB CV F1-score (spam) Mean: {np.mean(mnb_f1_scores):.4f}")
print(f"MultinomialNB CV F1-score (spam) Std:  {np.std(mnb_f1_scores):.4f}")

knn_f1_scores = []
knn = KNeighborsClassifier(n_neighbors=5)


for train_index, val_index in skf.split(X_train_selected_reset, y_train_reset):
    X_train_fold, X_val_fold = X_train_selected_reset.iloc[train_index], X_train_selected_reset.iloc[val_index]
    y_train_fold, y_val_fold = y_train_reset.iloc[train_index], y_train_reset.iloc[val_index]

    knn.fit(X_train_fold, y_train_fold)
    y_pred_knn = knn.predict(X_val_fold)

    score = f1_spam(y_val_fold, y_pred_knn)
    knn_f1_scores.append(score)

print(f"KNeighborsClassifier CV F1-score (spam) Mean: {np.mean(knn_f1_scores):.4f}")
print(f"KNeighborsClassifier CV F1-score (spam) Std:  {np.std(knn_f1_scores):.4f}")

print(f"Multinomial Naive Bayes (MNB) F1-score (spam) Mean: {np.mean(mnb_f1_scores):.4f}")
print(f"Multinomial Naive Bayes (MNB) F1-score (spam) Std:  {np.std(mnb_f1_scores):.4f}\n")

print(f"K-Nearest Neighbors (KNN) F1-score (spam) Mean: {np.mean(knn_f1_scores):.4f}")
print(f"K-Nearest Neighbors (KNN) F1-score (spam) Std:  {np.std(knn_f1_scores):.4f}")

print('NB ', mnb_f1_scores)
print('KNN ', knn_f1_scores)

t_statistic, p_value = ttest_rel(mnb_f1_scores, knn_f1_scores)

print(f"Paired t-test p-value:     {p_value:.4f}")

final_mnb_model = MultinomialNB()
final_mnb_model.fit(X_train_selected, y_train)

X_test_cleaned = X_test.apply(simple_clean)
print("First 5 entries of X_test_cleaned:")
print(X_test_cleaned.head())

X_test_manual_features = pd.DataFrame(X_test.apply(extract_custom_features).tolist())
print("First 5 entries of X_test_manual_features:")
print(X_test_manual_features.head())

X_test_bow = vectorizer.transform(X_test_cleaned)

X_test_bow_df = pd.DataFrame(
    X_test_bow.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=X_test_cleaned.index
)

print("First 5 entries of X_test_bow_df:")
print(X_test_bow_df.head())

X_test_manual_features = X_test.apply(extract_custom_features).apply(pd.Series)

X_test_combined = pd.concat([X_test_manual_features, X_test_bow_df], axis=1)

X_test_selected = selector.transform(X_test_combined)

# Ensure the output is a DataFrame with correct column names and index
X_test_selected = pd.DataFrame(
    X_test_selected,
    columns=selected_features,
    index=X_test_combined.index
)

print(f"Shape of X_test_selected: {X_test_selected.shape}")
print("\nFirst 5 entries of X_test_selected:")
print(X_test_selected.head())

y_pred_test = final_mnb_model.predict(X_test_selected)

accuracy_test = accuracy_score(y_test, y_pred_test)
precision_test = precision_score(y_test, y_pred_test, pos_label='spam', average='binary')
recall_test = recall_score(y_test, y_pred_test, pos_label='spam', average='binary')
f1_test = f1_spam(y_test, y_pred_test)

print(f"Accuracy on Test Set:  {accuracy_test:.4f}")
print(f"Precision on Test Set: {precision_test:.4f}")
print(f"Recall on Test Set:    {recall_test:.4f}")
print(f"F1-score (spam) on Test Set: {f1_test:.4f}")

print("Classification Report on Test Set:")
print(classification_report(y_test, y_pred_test))

plt.figure(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_test, cmap='Blues', normalize='true')
plt.title('Confusion Matrix on Test Set')
plt.grid(False)
plt.show()

First 5 entries of X_train_cleaned:
41327    i totally disagree with the person who first c...
42114    i saw this movie back in  on a double-bill wit...
20210    cheerleader massacre ()<br /><br />starring: t...
5576     i saw the greek tycoon when it first came out ...
26594    a kinda remake of planes trains and automobile...
Name: review, dtype: object
First 5 entries of X_train_manual_features:
     0  1  2  3
0  130  0  0  3
1   92  1  2  2
2  175  1  1  2
3  124  0  2  3
4  115  0  2  2
First 5 entries of X_train_bow_df:
       ___  ____  _____  ______  ____________________________________  \
41327    0     0      0       0                                     0   
42114    0     0      0       0                                     0   
20210    0     0      0       0                                     0   
5576     0     0      0       0                                     0   
26594    0     0      0       0                                     0   

       _absolute  _annie_  